# MineToday V137 — V132-Anchored Ordinal State-Utility Trust Region (AOSUTR)

Final-day research notebook implementing the frozen Post-V136 V137 contract. It reconstructs the V132-style policy anchor from organizer inputs, fits a fold-safe ordinal state prior, preserves Top1, reranks only anchor positions 2–8, and fails closed unless every pre-registered gate passes.

**Important:** this notebook deliberately does not read historical submission files or pseudo-ground-truth artifacts. Full training is intended for Kaggle; no local training is required.

In [1]:
# V137 — V132-Anchored Ordinal State-Utility Trust Region (AOSUTR)
# Runtime, immutable configuration, organizer-only data discovery.
import os, sys, re, io, gc, json, ast, html, math, time, hashlib, random, warnings, unicodedata, subprocess, platform
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict

os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("OMP_NUM_THREADS","4")
os.environ.setdefault("OPENBLAS_NUM_THREADS","4")
os.environ.setdefault("MKL_NUM_THREADS","4")
os.environ.setdefault("NUMEXPR_NUM_THREADS","4")
os.environ.setdefault("PYTHONHASHSEED","20260907")
warnings.filterwarnings("ignore")

def _ensure_runtime():
    import importlib.metadata as im
    need=[]
    try: im.version('lightgbm')
    except Exception: need.append('lightgbm==4.6.0')
    if need:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
_ensure_runtime()

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import lightgbm as lgb
from IPython.display import display

SEED=20260907
SECOND_SEED=20260917
STATE_SEED=20260909
FOLD_SEED=20260907
ALPHA_GRID=(0.0,0.01,0.02,0.05)
random.seed(SEED); np.random.seed(SEED)

@dataclass(frozen=True)
class V137Config:
    outer_folds:int=5
    inner_folds:int=3
    state_ridge_alpha:float=1000.0
    q12_ridge_alpha:float=100.0
    kmeans_clusters:int=6
    kmeans_n_init:int=20
    bootstraps:int=2000
    smoke:bool=bool(int(os.getenv('V137_SMOKE','0')))
    run_stress:bool=bool(int(os.getenv('V137_RUN_STRESS','1')))

VCFG=V137Config()
class _AnchorCfg:
    smoke=VCFG.smoke
cfg=_AnchorCfg()  # required by the frozen anchor engine only
OUT=Path('/kaggle/working/v137') if Path('/kaggle/working').exists() else Path('./v137')
OUT.mkdir(parents=True,exist_ok=True)
print('V137 config',asdict(VCFG),'alpha grid',ALPHA_GRID)
try: print(subprocess.check_output(['nvidia-smi','-L'],text=True).strip())
except Exception: print('GPU not visible; V137 itself is CPU-heavy and targets Kaggle T4x2 compatibility.')

REQUIRED={'chat_history.csv','modules_catalog.csv','user_assessments.csv','train_relevance.csv','test.csv','sample_submission.csv'}
def locate_data():
    for root in [Path('/kaggle/input'),Path('.'),Path('/mnt/data')]:
        if not root.exists(): continue
        cands=[root]
        try: cands += [p for p in root.rglob('*') if p.is_dir()]
        except Exception: pass
        for d in cands:
            try: names={p.name for p in d.iterdir() if p.is_file()}
            except Exception: continue
            if REQUIRED.issubset(names): return d
    raise FileNotFoundError('MineToday organizer CSV directory not found.')

DATA=locate_data(); print('DATA',DATA)
chat=pd.read_csv(DATA/'chat_history.csv')
modules=pd.read_csv(DATA/'modules_catalog.csv')
assess=pd.read_csv(DATA/'user_assessments.csv')
train=pd.read_csv(DATA/'train_relevance.csv')
test=pd.read_csv(DATA/'test.csv')
sample=pd.read_csv(DATA/'sample_submission.csv')
MODULES=list(train.columns[1:]); M=len(MODULES); NTR=len(train); NTE=len(test); N=NTR+NTE
assert M==17 and NTR==4000 and NTE==1000
assert MODULES==list(sample.columns[1:]) and test.user_id.astype(str).tolist()==sample.user_id.astype(str).tolist()
assert train.user_id.is_unique and test.user_id.is_unique and not set(train.user_id.astype(str))&set(test.user_id.astype(str))
ALL_USERS=train.user_id.astype(str).tolist()+test.user_id.astype(str).tolist(); UID2I={u:i for i,u in enumerate(ALL_USERS)}
y=train[MODULES].to_numpy(np.float32)
assert np.isfinite(y).all() and (y>=0).all()

DISCOUNT=1/np.log2(np.arange(2,7,dtype=np.float64))
def top5_idx(scores):
    s=np.asarray(scores); ix=np.argpartition(-s,4,axis=1)[:,:5]; loc=np.take_along_axis(s,ix,1)
    return np.take_along_axis(ix,np.argsort(-loc,axis=1,kind='stable'),1)
def per_user_ndcg(yy,ss):
    ix=top5_idx(ss); dcg=(np.take_along_axis(yy,ix,1)*DISCOUNT).sum(1)
    ideal=np.sort(yy,axis=1)[:,-5:][:,::-1]; idcg=(ideal*DISCOUNT).sum(1)
    return dcg/np.maximum(idcg,1e-12)
def ndcg5(yy,ss): return float(per_user_ndcg(yy,ss).mean())

def strata(rows):
    rows=np.asarray(rows,int); yy=y[rows]; top=np.argmax(yy,1); kk=(yy>0).sum(1)
    raw=np.asarray([f'{a}_{b}' for a,b in zip(top,kk)],object); c=Counter(raw)
    return np.asarray([x if c[x]>=5 else str(a) for x,a in zip(raw,top)],object)

def random_outer_splits():
    rows=np.arange(NTR); st=strata(rows)
    sk=StratifiedKFold(5,shuffle=True,random_state=FOLD_SEED)
    return [(rows[a],rows[b]) for a,b in sk.split(rows,st)]

def inner_splits(rows,seed):
    rows=np.asarray(rows,int); st=strata(rows)
    sk=StratifiedKFold(3,shuffle=True,random_state=int(seed))
    return [(rows[a],rows[b]) for a,b in sk.split(rows,st)]


V137 config {'outer_folds': 5, 'inner_folds': 3, 'state_ridge_alpha': 1000.0, 'q12_ridge_alpha': 100.0, 'kmeans_clusters': 6, 'kmeans_n_init': 20, 'bootstraps': 2000, 'smoke': False, 'run_stress': True} alpha grid (0.0, 0.01, 0.02, 0.05)
GPU 0: Tesla T4 (UUID: GPU-a99081a8-91be-cdba-9f53-eb6e4cb85486)
GPU 1: Tesla T4 (UUID: GPU-e879cc8a-025b-1e8d-483b-b92bcec2c6ec)
DATA /kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026


In [2]:
# Frozen V132-style policy-anchor engine (organizer-derived reconstruction)
# =========================
# Reused from the executed V132 research notebook; no historical prediction bytes are loaded.
# It reconstructs the V30/V31 + B34/B45 policy geometry inside arbitrary train/validation splits.
all_users=ALL_USERS; uid2idx=UID2I; assessments=assess
N_MODULES=M; N_TRAIN=NTR; N_TEST=NTE
RIDGE_ALPHA=3.0; REFERENCE_W=0.45
ANCHOR_INNER_FOLDS=2 if cfg.smoke else 4
INTENT_INNER_FOLDS=2 if cfg.smoke else 4
LGB_ESTIMATORS=40 if cfg.smoke else 1200
LGB_EARLY_STOP=5 if cfg.smoke else 80
RANKER_ESTIMATORS=40 if cfg.smoke else 360
RANKER_LEARNING_RATE=0.04 if cfg.smoke else 0.025
ALPHA=0.16
INTENT_PARENT_STRENGTH=40.0; INTENT_CHILD_STRENGTH=30.0
BOOTSTRAPS=100 if cfg.smoke else 1200
EXPECTED_BOUNDARY_COUNTS={"B23":9,"B34":10,"B45":8}

def _seed_from_text(s): return int(hashlib.sha1(s.encode("utf-8")).hexdigest()[:8],16)%(2**31-1)
top5_indices=top5_idx
Y_BIN=(y>0).astype(np.float32)
Y_TOP=np.zeros_like(y,np.float32); Y_TOP[np.arange(NTR),np.argmax(y,axis=1)]=1.0

# Robust assessment parsing — exact V3.1 mechanism
WORD_NUM = {"nol":0, "satu":1, "dua":2, "tiga":3, "empat":4, "lima":5}
SKILL_NAMES = ["python","sql","statistics","eda","ml","ml_eval","dl","genai","business","research"]

A = np.zeros((len(all_users), 10), np.float32)
A_INVALID = np.zeros_like(A)

for row in assessments.itertuples(index=False):
    try:
        d = json.loads(row.assessment_result)
    except Exception:
        d = ast.literal_eval(row.assessment_result)

    vals, invalid = [], []
    for v in d.values():
        sv = str(v).strip().lower()
        if sv in WORD_NUM:
            value, bad = float(WORD_NUM[sv]), 0
        else:
            try:
                value = float(v)
            except Exception:
                value = np.nan
            bad = int((not np.isfinite(value)) or value < 0 or value > 5)
            if bad:
                value = np.nan
        vals.append(value)
        invalid.append(bad)

    vals = np.asarray(vals, float)
    med = float(np.nanmedian(vals)) if np.isfinite(vals).any() else 2.0
    vals = np.where(np.isfinite(vals), vals, med)
    i = uid2idx[row.user_id]
    A[i] = vals
    A_INVALID[i] = invalid

print("invalid assessment cells:", int(A_INVALID.sum()), "/", A_INVALID.size)

# Exact V3.1 dataset-derived topic/style parser
TOPIC_PATTERNS = {
    "excel": r"\b(?:ms\.?\s*excel|excel|pivot(?: table)?|vlookup|hlookup|spreadsheet)\b",
    "python": r"\b(?:python|pyton|pithon|pandas(?:\s*&\s*| dan |\s+)?numpy|pandas|numpy)\b",
    "sql": r"\b(?:sql|escuel|query database|join tabel|database relasional|window function)\b",
    "statistics": r"\b(?:statistik(?:\s*&\s*probabilitas)?|probabilitas|a\s*/?\s*b testing|uji hipotesis|distribusi data)\b",
    "eda": r"\b(?:eda|exploratory data analysis|data cleaning|missing values?|matplotlib|seaborn)\b",
    "ml_eval": r"\b(?:evaluasi model(?: ml)?|metrik model|metric model|evaluasi performa model)\b",
    "mlops": r"\b(?:mlops|deployment model|deploy(?:ment)? model|ci\s*/?\s*cd|monitoring model|serving model|fastapi)\b",
    "cv": r"\b(?:computer vision|cnn|deep learning|deteksi objek|klasifikasi gambar|image classification)\b",
    "nlp": r"\b(?:natural language processing|nlp|analisis sentimen|sentiment analysis|word2vec|tf\s*-?\s*idf|transformer)\b",
    "genai": r"\b(?:generative ai|genai|large language model|llm|rag|chatgpt|chatbot|fine[- ]?tuning model bahasa)\b",
    "ml": r"\b(?:machine learning|bikin model ml|model machine learning|regresi\s*(?:&|dan)\s*klasifikasi|clustering|supervised|unsupervised|hyperparameter)\b",
    "business": r"\b(?:translate masalah bisnis ke data|menerjemahkan masalah bisnis|business(?:-| )to(?:-| )ds|permasalahan bisnis)\b",
    "dashboard": r"\b(?:dashboard|looker studio|tableau|visualisasi interaktif|storytelling data)\b",
    "webscrape": r"\b(?:web scrap(?:e|ing)|scraping|beautifulsoup|selenium|ekstraksi data.*website)\b",
    "git": r"\b(?:version control|git(?:hub|lab)?|repository|repo kode)\b",
    "prompt": r"\b(?:prompt engineering|prompting|merancang prompt|optimasi prompt)\b",
    "automation": r"\b(?:ai automation|otomasi|otomatis|automation|workflow|n8n)\b",
    "nocode": r"\b(?:no[ -]?code|zero coding|tanpa perlu (?:belajar )?(?:coding|koding)|gausah (?:coding|koding)|gak perlu (?:coding|koding)|tanpa (?:coding|koding))\b",
    "career": r"\b(?:portofolio|portfolio|cv|test interview|interview|karir|career|prospek kerja|negosiasi)\b",
}
TOPIC_RX = {k: re.compile(v, re.I) for k,v in TOPIC_PATTERNS.items()}
TOPICS = list(TOPIC_PATTERNS)

NOISE_RX = re.compile(
    r"reset password|email verifikasi|sertifikat kelulusan|pembayaran|cicilan|invoice|"
    r"akun.*(?:error|masalah)|lms.*(?:error|masalah)", re.I
)
STYLE_PATTERNS = {
    "beginner": r"pemula|dari nol|gaptek|belum pernah (?:ngoding|coding|koding)|background(?:nya)? bukan (?:it|data)|banting setir",
    "advanced": r"udah lumayan paham|sudah lumayan paham|stuck|prasyarat|syarat|\breq\b|basic apa|lanjutin",
    "urgent": r"urgent|secepatnya|sefastnya|intensif|seminggu|minggu depan",
    "budget": r"budget|diskon|worth it|cekak|fomo|kompetitor",
    "online": r"online|durasi|kapan buka|mulai minggu|jadwal",
    "nocode_style": r"tanpa perlu.*(?:coding|koding)|gausah.*(?:coding|koding)|gak perlu.*(?:coding|koding)|otomatis.*ai|solusi otomatis",
    "career_style": r"portofolio|portfolio|test interview|prospek kerja|karir|career",
}
STYLE_RX = {k: re.compile(v, re.I) for k,v in STYLE_PATTERNS.items()}
STYLES = list(STYLE_PATTERNS)

def clean_text(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " <url> ", text)
    text = re.sub(r"[^\w\s<>/&+-]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()

def extract_topics(text):
    return [k for k,rx in TOPIC_RX.items() if rx.search(str(text))]

def make_skeleton(text):
    s = clean_text(text)
    for rx in TOPIC_RX.values():
        s = rx.sub(" <topic> ", s)
    s = re.sub(r"\b\d+\b", " <num> ", s)
    return re.sub(r"\s+", " ", s).strip()

def template_token(skeleton):
    return "tpl_" + hashlib.md5(skeleton.encode("utf-8")).hexdigest()[:10]

chat["timestamp"] = pd.to_datetime(chat["timestamp"], errors="coerce")
chat["clean"] = chat["user_chat_text"].fillna("").map(clean_text)
chat["is_noise"] = chat["user_chat_text"].fillna("").map(lambda x: bool(NOISE_RX.search(str(x))))
chat["topics"] = chat["user_chat_text"].fillna("").map(extract_topics)
chat["skeleton"] = chat["user_chat_text"].fillna("").map(make_skeleton)
for name,rx in STYLE_RX.items():
    chat["sty_"+name] = chat["user_chat_text"].fillna("").map(lambda x,r=rx: int(bool(r.search(str(x)))))

chat = chat.sort_values(["user_id","timestamp","chat_id"]).reset_index(drop=True)
user_chat = {u:g.copy() for u,g in chat.groupby("user_id", sort=False)}
print("administrative/noise rows:", int(chat["is_noise"].sum()), "/", len(chat))

# V3.1 template-aware sparse representation + structured user features
USER_FEATURES = np.zeros((len(all_users), 10 + 10 + 5 + len(STYLES)*2 + len(TOPICS)*5), np.float32)

def build_user(u):
    ui = uid2idx[u]
    g = user_chat.get(u)
    if g is None:
        learn = pd.DataFrame(columns=chat.columns)
        g = learn
    else:
        learn = g[~g["is_noise"]].copy()
        if len(learn) == 0:
            learn = g.copy()

    parts = []
    n = len(learn)
    for j,row in enumerate(learn.itertuples(index=False)):
        distance = n - 1 - j
        tag = "current" if distance == 0 else ("recent" if distance <= 2 else "older")
        parts.append(f"[{tag}] {row.clean}")
        parts.append(template_token(row.skeleton))
        for tp in row.topics:
            parts.append("topic_" + tp)
            if distance == 0:
                parts.extend(["latest_topic_" + tp] * 3)
            elif distance <= 2:
                parts.extend(["recent_topic_" + tp] * 2)
        for sty in STYLES:
            if getattr(row, "sty_"+sty):
                parts.append("style_" + sty)

    for j,skill in enumerate(SKILL_NAMES):
        v = int(round(float(A[ui,j])))
        band = "low" if v <= 1 else ("mid" if v <= 3 else "high")
        parts += [f"skill_{skill}_{v}", f"skill_{skill}_bin_{band}"]

    feats = list(A[ui]) + list(A_INVALID[ui])
    total = len(g)
    learning_count = len(learn)
    noise_count = int(g["is_noise"].sum()) if total else 0
    span = 0.0
    if total >= 2 and g["timestamp"].notna().sum() >= 2:
        span = float((g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 86400.0)
    feats += [total, learning_count, noise_count, span, min(total,6)]

    for sty in STYLES:
        feats.append(float(learn["sty_"+sty].sum()) if learning_count else 0.0)
    for sty in STYLES:
        feats.append(float(learn.iloc[-1]["sty_"+sty]) if learning_count else 0.0)

    rows = list(learn.itertuples(index=False))
    nfull = len(rows)
    for tp in TOPICS:
        count = rec_count = last_pos = 0.0
        latest = 0.0
        for j,row in enumerate(rows):
            if tp in row.topics:
                count += 1.0
                rec_count += float(np.exp(-0.7 * (nfull - 1 - j)))
                last_pos = max(last_pos, (j+1) / max(nfull,1))
        if rows and tp in rows[-1].topics:
            latest = 1.0
        feats += [count, rec_count, latest, last_pos, float(count > 0)]

    USER_FEATURES[ui] = np.asarray(feats, np.float32)
    return " ".join(parts)

texts = np.asarray([build_user(u) for u in all_users], dtype=object)
assert USER_FEATURES.shape == (5000, 134)
assert np.isfinite(USER_FEATURES).all()
print("USER_FEATURES:", USER_FEATURES.shape)
print("example augmented text:", texts[1][:500])


# V42 target-blind sparse text basis. Full 5,000-user vocabulary matches the frozen lineage.
word_vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.995,sublinear_tf=True,strip_accents="unicode",max_features=(8000 if cfg.smoke else 50000),dtype=np.float32)
char_vec=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,sublinear_tf=True,max_features=(12000 if cfg.smoke else 60000),dtype=np.float32)
X_WORD=word_vec.fit_transform(texts); X_CHAR=char_vec.fit_transform(texts); X=hstack([X_WORD,X_CHAR],format="csr")
CONSENSUS3_OVERRIDES={"M_002":0.44,"M_006":0.55,"M_007":0.60,"M_010":0.40,"M_011":0.55,"M_012":0.55,"M_015":0.44,"M_017":0.53}
def consensus3_vector():
    w=np.full(N_MODULES,REFERENCE_W,np.float32)
    for mid,val in CONSENSUS3_OVERRIDES.items(): w[MODULES.index(mid)]=float(val)
    return w
CONS_W=consensus3_vector()

# Module-native lexical / intent / readiness surfaces inherited from V30 Base.
MODULE_TEXT = (
    modules.set_index("module_id").loc[MODULES, "module_name"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "description_and_syllabus"].astype(str) + " " +
    modules.set_index("module_id").loc[MODULES, "prerequisite_level"].astype(str)
).tolist()
MODULE_WORD = word_vec.transform(MODULE_TEXT)
MODULE_CHAR = char_vec.transform(MODULE_TEXT)
LEX_WORD = (X_WORD @ MODULE_WORD.T).toarray().astype(np.float32)
LEX_CHAR = (X_CHAR @ MODULE_CHAR.T).toarray().astype(np.float32)

MODULE_TOPIC_MAP = {
    "M_001":["excel"], "M_002":["python"], "M_003":["sql"],
    "M_004":["webscrape","python"], "M_005":["git"], "M_006":["statistics"],
    "M_007":["eda","python","statistics"], "M_008":["dashboard","sql","excel"],
    "M_009":["ml","ml_eval","eda","statistics"], "M_010":["cv","ml"],
    "M_011":["nlp","ml"], "M_012":["genai","nlp"], "M_013":["prompt","genai"],
    "M_014":["automation","genai"], "M_015":["nocode","genai"],
    "M_016":["mlops","ml","python","git"], "M_017":["career"],
}
MODULE_SKILL_MAP = {
    "M_001":[], "M_002":[0], "M_003":[1], "M_004":[0], "M_005":[9],
    "M_006":[2], "M_007":[0,2,3], "M_008":[1], "M_009":[2,3,4,5],
    "M_010":[4,6], "M_011":[4], "M_012":[4,7], "M_013":[7],
    "M_014":[7,8], "M_015":[7], "M_016":[0,4,5,9], "M_017":[8,9],
}

MODULE_COUNT = np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_RECENCY = np.zeros_like(MODULE_COUNT)
MODULE_LATEST = np.zeros_like(MODULE_COUNT)
LATEST_GROUP = np.empty(len(all_users),dtype=object); LATEST_GROUP[:] = "no_chat"
LATEST_INTENT_KEY = np.empty(len(all_users),dtype=object); LATEST_INTENT_KEY[:] = "intent:no_chat"

for u in all_users:
    ui=uid2idx[u]
    g=user_chat.get(u)
    if g is None or len(g)==0:
        continue
    learn=g[~g["is_noise"]].copy()
    if len(learn)==0:
        learn=g.copy()
    rows=list(learn.itertuples(index=False))
    if rows:
        LATEST_GROUP[ui]=rows[-1].skeleton or "empty"
        tps=sorted(set(rows[-1].topics))
        LATEST_INTENT_KEY[ui] = "intent:" + ("+".join(tps) if tps else "none")
    n=len(rows)
    for j,row in enumerate(rows):
        dist=n-1-j; w=float(np.exp(-0.7*dist)); row_topics=set(row.topics)
        for m,mid in enumerate(MODULES):
            hits=sum(tp in row_topics for tp in MODULE_TOPIC_MAP[mid])
            if hits:
                MODULE_COUNT[ui,m]+=hits
                MODULE_RECENCY[ui,m]+=w*hits
                if dist==0:
                    MODULE_LATEST[ui,m]=1.0

MODULE_READINESS=np.zeros((len(all_users),N_MODULES),np.float32)
MODULE_INVALID=np.zeros_like(MODULE_READINESS)
for m,mid in enumerate(MODULES):
    idx=MODULE_SKILL_MAP[mid]
    if idx:
        MODULE_READINESS[:,m]=A[:,idx].mean(1)
        MODULE_INVALID[:,m]=A_INVALID[:,idx].mean(1)
    else:
        MODULE_READINESS[:,m]=A.mean(1)
        MODULE_INVALID[:,m]=A_INVALID.mean(1)

# Fixed readiness discretization consistent with the V3.1 low/mid/high assessment bins.
_ready_round=np.floor(MODULE_READINESS+0.5)
MODULE_READY_BAND=np.where(_ready_round<=1,0,np.where(_ready_round<=3,1,2)).astype(np.int8)

TRAIN_GROUP=np.asarray(LATEST_GROUP[:N_TRAIN],dtype=object)
TRAIN_INTENT=np.asarray(LATEST_INTENT_KEY[:N_TRAIN],dtype=object)
TEST_INTENT=np.asarray(LATEST_INTENT_KEY[N_TRAIN:],dtype=object)

prtxt=modules.set_index("module_id")["prerequisite_level"].astype(str).to_dict()
def module_level(mid):
    s=prtxt[mid].lower()
    if "pemula" in s: return 0
    if "menengah" in s: return 1
    if "lanjutan" in s: return 2
    if "ahli" in s: return 3
    return 4
MODULE_LEVEL=np.asarray([module_level(m) for m in MODULES],np.int16)
MODULE_LEVEL_ONEHOT=np.eye(5,dtype=np.float32)[MODULE_LEVEL]

# Ordinal LambdaRank labels: 0=irrelevant, 1=#6 ... 6=#1.
ORDINAL_GRADES=np.zeros_like(y,dtype=np.int16)
for i,row in enumerate(y):
    pos=np.flatnonzero(row>0)
    order=pos[np.argsort(-row[pos],kind="mergesort")]
    for rank,m in enumerate(order,1):
        ORDINAL_GRADES[i,m]=max(1,7-rank)
LABEL_GAIN=[0.0,0.25,0.40,0.55,0.70,0.85,1.00]

def user_strata_from_y(yy):
    yy=np.asarray(yy)
    k=(yy>0).sum(1).astype(int)
    top=np.argmax(yy,axis=1).astype(int)
    return (top*4+np.clip(k-3,0,3)).astype(int)

def make_group_folds_local(yy,groups,n_splits,seed):
    yy=np.asarray(yy); groups=np.asarray(groups,dtype=object)
    ns=min(int(n_splits),len(np.unique(groups)))
    if ns<2:
        sk=StratifiedKFold(n_splits=2,shuffle=True,random_state=int(seed))
        return list(sk.split(np.zeros(len(yy)),user_strata_from_y(yy)))
    sg=StratifiedGroupKFold(n_splits=ns,shuffle=True,random_state=int(seed))
    folds=list(sg.split(np.zeros(len(yy)),user_strata_from_y(yy),groups=groups))
    for tr,va in folds:
        assert len(set(groups[tr]).intersection(set(groups[va])))==0
    return folds

def make_group_folds_global(rows,n_splits,seed):
    rows=np.asarray(rows,int)
    local=make_group_folds_local(y[rows],TRAIN_GROUP[rows],n_splits,seed)
    return [(rows[tr],rows[va]) for tr,va in local]

print("catalog lexical matrices:",LEX_WORD.shape,LEX_CHAR.shape)
print("latest template groups:",len(np.unique(TRAIN_GROUP)))
print("latest intent signatures train/test:",len(np.unique(TRAIN_INTENT)),len(np.unique(TEST_INTENT)))
print("test intent signatures unseen in train:",len(set(TEST_INTENT)-set(TRAIN_INTENT)))
print("readiness band counts:",dict(zip(*np.unique(MODULE_READY_BAND[:N_TRAIN],return_counts=True))))
print("ordinal grade counts:",dict(zip(*np.unique(ORDINAL_GRADES,return_counts=True))))

ALL_INTENT=np.concatenate([np.asarray(TRAIN_INTENT,dtype=object),np.asarray(TEST_INTENT,dtype=object)])
# Strict fold-safe hierarchical Intent-State target encoding.
INTENTSTATE_KEYS = [
    "IS_PARENT_SEEN","IS_PARENT_SUPPORT_LOG","IS_PARENT_CONF",
    "IS_PARENT_POS","IS_PARENT_ORD","IS_PARENT_RAW",
    "IS_CHILD_SUPPORT_LOG","IS_CHILD_CONF",
    "IS_CHILD_POS","IS_CHILD_ORD","IS_CHILD_RAW","IS_CHILD_TOP5",
    "IS_POS_LOG_LIFT_GLOBAL","IS_ORD_DELTA_GLOBAL","IS_ORD_DELTA_PARENT","IS_RAW_DELTA_PARENT",
]

def _intent_targets(y_fit,grades_fit):
    y_fit=np.asarray(y_fit,np.float32); grades_fit=np.asarray(grades_fit,np.int16)
    return {
        "pos":(y_fit>0).astype(np.float32),
        "ord":(grades_fit.astype(np.float32)/6.0),
        "raw":y_fit.astype(np.float32),
        "top5":(grades_fit>=2).astype(np.float32),
    }

def fit_intentstate_bundle(y_fit,grades_fit,intent_fit,ready_band_fit):
    y_fit=np.asarray(y_fit,np.float32)
    grades_fit=np.asarray(grades_fit,np.int16)
    intent_fit=np.asarray(intent_fit,dtype=object)
    ready_band_fit=np.asarray(ready_band_fit,np.int8)
    n=len(y_fit)
    assert grades_fit.shape==y_fit.shape==ready_band_fit.shape
    T=_intent_targets(y_fit,grades_fit)
    glob={k:v.mean(0).astype(np.float32) for k,v in T.items()}

    parent={}
    child={}
    for key in np.unique(intent_fit):
        idx=np.flatnonzero(intent_fit==key)
        ns=int(len(idx))
        vals={}
        for name,M in T.items():
            sm=(M[idx].sum(0)+INTENT_PARENT_STRENGTH*glob[name])/(ns+INTENT_PARENT_STRENGTH)
            vals[name]=sm.astype(np.float32)
        parent[str(key)]={"n":ns,**vals}

        for m in range(N_MODULES):
            rb=ready_band_fit[idx,m]
            for b in (0,1,2):
                local=idx[rb==b]
                nc=int(len(local))
                if nc==0:
                    continue
                cv={}
                for name,M in T.items():
                    prior=float(vals[name][m])
                    cv[name]=float((M[local,m].sum()+INTENT_CHILD_STRENGTH*prior)/(nc+INTENT_CHILD_STRENGTH))
                child[(str(key),m,int(b))]={"n":nc,**cv}

    return {"n_fit":n,"global":glob,"parent":parent,"child":child}

def apply_intentstate_bundle(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object)
    ready_band=np.asarray(ready_band,np.int8)
    n=len(intent_keys)
    assert ready_band.shape==(n,N_MODULES)
    out={k:np.zeros((n,N_MODULES),np.float32) for k in INTENTSTATE_KEYS}
    glob=bundle["global"]
    logden=np.log1p(max(1,int(bundle["n_fit"])))

    for i,key0 in enumerate(intent_keys):
        key=str(key0)
        p=bundle["parent"].get(key)
        parent_seen=float(p is not None)
        if p is None:
            pn=0
            pvals={name:glob[name] for name in ("pos","ord","raw","top5")}
        else:
            pn=int(p["n"])
            pvals={name:p[name] for name in ("pos","ord","raw","top5")}
        pconf=pn/(pn+INTENT_PARENT_STRENGTH) if pn else 0.0
        pslog=np.log1p(pn)/logden if logden>0 else 0.0

        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            if c is None:
                cn=0
                cpos=float(pvals["pos"][m]); cord=float(pvals["ord"][m])
                craw=float(pvals["raw"][m]); ctop5=float(pvals["top5"][m])
            else:
                cn=int(c["n"]); cpos=float(c["pos"]); cord=float(c["ord"])
                craw=float(c["raw"]); ctop5=float(c["top5"])
            cconf=cn/(cn+INTENT_CHILD_STRENGTH) if cn else 0.0
            cslog=np.log1p(cn)/logden if logden>0 else 0.0
            gpos=float(glob["pos"][m]); gord=float(glob["ord"][m]); graw=float(glob["raw"][m])
            pord=float(pvals["ord"][m]); praw=float(pvals["raw"][m])

            out["IS_PARENT_SEEN"][i,m]=parent_seen
            out["IS_PARENT_SUPPORT_LOG"][i,m]=pslog
            out["IS_PARENT_CONF"][i,m]=pconf
            out["IS_PARENT_POS"][i,m]=float(pvals["pos"][m])
            out["IS_PARENT_ORD"][i,m]=pord
            out["IS_PARENT_RAW"][i,m]=praw
            out["IS_CHILD_SUPPORT_LOG"][i,m]=cslog
            out["IS_CHILD_CONF"][i,m]=cconf
            out["IS_CHILD_POS"][i,m]=cpos
            out["IS_CHILD_ORD"][i,m]=cord
            out["IS_CHILD_RAW"][i,m]=craw
            out["IS_CHILD_TOP5"][i,m]=ctop5
            out["IS_POS_LOG_LIFT_GLOBAL"][i,m]=float(np.clip(np.log((cpos+1e-4)/(gpos+1e-4)),-2.0,2.0))
            out["IS_ORD_DELTA_GLOBAL"][i,m]=cord-gord
            out["IS_ORD_DELTA_PARENT"][i,m]=cord-pord
            out["IS_RAW_DELTA_PARENT"][i,m]=craw-praw

    for k,v in out.items():
        if not np.isfinite(v).all():
            raise ValueError(f"non-finite Intent-State feature {k}")
    return out

def crossfit_intentstate_features(y_local,grades_local,intent_local,ready_band_local,groups_local,seed):
    y_local=np.asarray(y_local,np.float32)
    grades_local=np.asarray(grades_local,np.int16)
    intent_local=np.asarray(intent_local,dtype=object)
    ready_band_local=np.asarray(ready_band_local,np.int8)
    groups_local=np.asarray(groups_local,dtype=object)
    out={k:np.zeros_like(y_local,np.float32) for k in INTENTSTATE_KEYS}
    folds=make_group_folds_local(y_local,groups_local,INTENT_INNER_FOLDS,seed)
    seen=np.zeros(len(y_local),bool)
    for tr,va in folds:
        assert len(set(groups_local[tr]).intersection(set(groups_local[va])))==0
        bundle=fit_intentstate_bundle(y_local[tr],grades_local[tr],intent_local[tr],ready_band_local[tr])
        vv=apply_intentstate_bundle(bundle,intent_local[va],ready_band_local[va])
        for k in INTENTSTATE_KEYS:
            out[k][va]=vv[k]
        seen[va]=True
    assert seen.all(), "Intent-State crossfit did not cover every training row"
    return out

def intentstate_support_audit(bundle,intent_keys,ready_band):
    intent_keys=np.asarray(intent_keys,dtype=object); ready_band=np.asarray(ready_band,np.int8)
    supports=[]; parent_support=[]; parent_seen=[]
    for i,key0 in enumerate(intent_keys):
        key=str(key0); p=bundle["parent"].get(key)
        pn=int(p["n"]) if p is not None else 0
        for m in range(N_MODULES):
            c=bundle["child"].get((key,m,int(ready_band[i,m])))
            supports.append(int(c["n"]) if c is not None else 0)
            parent_support.append(pn); parent_seen.append(int(p is not None))
    s=np.asarray(supports,int); ps=np.asarray(parent_support,int); pseen=np.asarray(parent_seen,int)
    rows=[]
    for thr in [1,5,10,20,30,50]:
        rows.append({"level":"child","threshold":thr,"coverage":float(np.mean(s>=thr))})
    rows.append({"level":"child","threshold":-1,"coverage":1.0,"median_support":float(np.median(s)),"p10_support":float(np.quantile(s,.10)),"p25_support":float(np.quantile(s,.25))})
    rows.append({"level":"parent","threshold":1,"coverage":float(np.mean(pseen>0)),"median_support":float(np.median(ps))})
    return pd.DataFrame(rows)

print("strict nested Intent-State TE helpers ready; features:",len(INTENTSTATE_KEYS))

def rank_matrix(scores):
    order=np.argsort(-np.asarray(scores),axis=1,kind="mergesort")
    ranks=np.empty_like(order,dtype=np.int16)
    ranks[np.arange(len(order))[:,None],order]=np.arange(1,N_MODULES+1,dtype=np.int16)[None,:]
    return ranks

def top5_order_change_rate(a,b):
    return float(np.mean(np.any(top5_indices(a)!=top5_indices(b),axis=1)))

def build_long_matrix_v31(rows_global,anchor,base,meta,intentstate=None):
    rows_global=np.asarray(rows_global,int); anchor=np.asarray(anchor,np.float32)
    base=np.asarray(base,np.float32); meta=np.asarray(meta,np.float32); n=len(rows_global)

    ranks=rank_matrix(anchor).astype(np.float32)
    top1=anchor.max(1,keepdims=True)
    sorted_scores=np.sort(anchor,axis=1)[:,::-1]
    cut5=sorted_scores[:,4:5]
    mu=anchor.mean(1,keepdims=True); sd=np.maximum(anchor.std(1,keepdims=True),1e-4)
    z=(anchor-mu)/sd
    order=np.argsort(-anchor,axis=1,kind="mergesort")
    prev_gap=np.zeros_like(anchor); next_gap=np.zeros_like(anchor)
    for i in range(n):
        oo=order[i]; ss=anchor[i,oo]
        pg=np.r_[0.0,ss[:-1]-ss[1:]]
        ng=np.r_[ss[:-1]-ss[1:],0.0]
        prev_gap[i,oo]=pg; next_gap[i,oo]=ng

    base_mats=[
        anchor, ranks/float(N_MODULES), (ranks<=5).astype(np.float32), top1-anchor, anchor-cut5,
        prev_gap,next_gap,z,base,meta,meta-base,
        LEX_WORD[rows_global],LEX_CHAR[rows_global],MODULE_COUNT[rows_global],MODULE_RECENCY[rows_global],
        MODULE_LATEST[rows_global],MODULE_READINESS[rows_global],MODULE_INVALID[rows_global],
    ]
    base_names=[
        "anchor_score","anchor_rank_norm","anchor_top5","top1_minus_score","score_minus_cut5",
        "prev_gap","next_gap","anchor_z","base_score","meta_score","meta_minus_base",
        "lex_word","lex_char","topic_count","topic_recency","topic_latest","readiness","invalid_readiness",
    ]
    pieces=[np.column_stack([m.reshape(-1) for m in base_mats]).astype(np.float32)]
    names=list(base_names)

    pieces.append(np.repeat(USER_FEATURES[rows_global],N_MODULES,axis=0).astype(np.float32))
    names += [f"user_f{i:03d}" for i in range(USER_FEATURES.shape[1])]
    pieces.append(np.tile(np.eye(N_MODULES,dtype=np.float32),(n,1)))
    names += [f"module_{m}" for m in MODULES]
    pieces.append(np.tile(MODULE_LEVEL_ONEHOT,(n,1)).astype(np.float32))
    names += [f"module_level_{i}" for i in range(5)]
    bounds={"base":sum(p.shape[1] for p in pieces)}

    if intentstate is not None:
        mats=[intentstate[k] for k in INTENTSTATE_KEYS]
        pieces.append(np.column_stack([m.reshape(-1) for m in mats]).astype(np.float32))
        names += [k.lower() for k in INTENTSTATE_KEYS]
        bounds["intentstate"]=sum(p.shape[1] for p in pieces)

    XX=np.hstack(pieces).astype(np.float32,copy=False)
    if XX.shape[0] != n*N_MODULES:
        raise AssertionError(XX.shape)
    if len(names)!=XX.shape[1]:
        raise AssertionError((len(names),XX.shape))
    if not np.isfinite(XX).all():
        raise ValueError("non-finite V31 long features")
    return XX,names,bounds

def make_ranker(seed):
    return lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=RANKER_ESTIMATORS,
        learning_rate=RANKER_LEARNING_RATE,
        num_leaves=15,
        min_child_samples=80,
        reg_alpha=1.0,
        reg_lambda=10.0,
        colsample_bytree=0.78,
        subsample=0.90,
        subsample_freq=1,
        max_bin=127,
        lambdarank_truncation_level=8,
        label_gain=LABEL_GAIN,
        random_state=int(seed),
        n_jobs=4,
        verbosity=-1,
        force_col_wise=True,
    )

def fit_ranker_predict(Xtr,Xva,grades_tr,n_users,seed):
    model=make_ranker(seed)
    group=np.full(int(n_users),N_MODULES,dtype=np.int32)
    model.fit(Xtr,grades_tr.reshape(-1).astype(np.int32),group=group)
    pred=model.predict(Xva).reshape(-1,N_MODULES).astype(np.float32)
    return model,pred

def align_ranker_to_anchor(raw_ranker,anchor):
    raw=np.asarray(raw_ranker,np.float32); anchor=np.asarray(anchor,np.float32)
    rmu=raw.mean(1,keepdims=True); rsd=np.maximum(raw.std(1,keepdims=True),1e-5)
    amu=anchor.mean(1,keepdims=True); asd=np.maximum(anchor.std(1,keepdims=True),1e-5)
    return (amu+(raw-rmu)*(asd/rsd)).astype(np.float32)

def blend_residual(anchor,raw_ranker,alpha=ALPHA):
    aligned=align_ranker_to_anchor(raw_ranker,anchor)
    return (anchor+float(alpha)*(aligned-anchor)).astype(np.float32)

def cluster_boot_ci(values,groups,seed,B=BOOTSTRAPS):
    values=np.asarray(values,float); groups=np.asarray(groups,dtype=object)
    ug=np.unique(groups); by={g:np.flatnonzero(groups==g) for g in ug}
    rng=np.random.default_rng(int(seed)); z=np.empty(int(B),float)
    for b in range(int(B)):
        sampled=rng.choice(ug,size=len(ug),replace=True)
        idx=np.concatenate([by[g] for g in sampled])
        z[b]=values[idx].mean()
    return tuple(float(x) for x in np.quantile(z,[.025,.975]))

print("V30-control / V31-IntentState LambdaRank helpers ready")

def _make_cal_local(seed,n_estimators=None):
    return lgb.LGBMClassifier(
        n_estimators=int(LGB_ESTIMATORS if n_estimators is None else n_estimators),
        learning_rate=.03,num_leaves=15,min_child_samples=50,
        reg_alpha=.5,reg_lambda=5.,colsample_bytree=.7,
        random_state=int(seed),n_jobs=4,verbosity=-1
    )


def build_anchor_split_full(train_rows,val_rows,seed):
    train_rows=np.asarray(train_rows,int); val_rows=np.asarray(val_rows,int)
    local_y=y[train_rows]; local_top=Y_TOP[train_rows]; local_bin=Y_BIN[train_rows]
    folds=make_group_folds_local(local_y,TRAIN_GROUP[train_rows],ANCHOR_INNER_FOLDS,seed)
    n=len(train_rows)

    raw=np.zeros((n,N_MODULES),np.float32); top=np.zeros_like(raw)
    for lt,lv in folds:
        mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
        mr.fit(X[train_rows[lt]],local_y[lt]); mt.fit(X[train_rows[lt]],local_top[lt])
        raw[lv]=mr.predict(X[train_rows[lv]]).astype(np.float32)
        top[lv]=mt.predict(X[train_rows[lv]]).astype(np.float32)
    mr=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mt=Ridge(alpha=RIDGE_ALPHA,solver="lsqr",tol=2e-3,max_iter=1000)
    mr.fit(X[train_rows],local_y); mt.fit(X[train_rows],local_top)
    base_o=(.9*raw+.1*top).astype(np.float32)
    base_v=(.9*mr.predict(X[val_rows])+.1*mt.predict(X[val_rows])).astype(np.float32)

    meta_o=np.zeros_like(base_o); meta_v=np.zeros((len(val_rows),N_MODULES),np.float32)
    for m,mid in enumerate(MODULES):
        xt=np.column_stack([base_o[:,m],USER_FEATURES[train_rows]]).astype(np.float32)
        xv=np.column_stack([base_v[:,m],USER_FEATURES[val_rows]]).astype(np.float32)
        yy=local_bin[:,m].astype(np.int32); its=[]
        for f,(lt,lv) in enumerate(folds):
            clf=_make_cal_local(seed+31*m+f)
            clf.fit(xt[lt],yy[lt],eval_set=[(xt[lv],yy[lv])],callbacks=[lgb.early_stopping(LGB_EARLY_STOP,verbose=False)])
            it=int(clf.best_iteration_ or LGB_ESTIMATORS); its.append(it)
            meta_o[lv,m]=clf.predict_proba(xt[lv],num_iteration=it)[:,1].astype(np.float32)
        final_it=max(20,int(np.median(its)))
        full=_make_cal_local(seed+997+31*m,final_it); full.fit(xt,yy)
        meta_v[:,m]=full.predict_proba(xv)[:,1].astype(np.float32)

    cons_o=((1.0-CONS_W)*base_o+CONS_W*meta_o).astype(np.float32)
    cons_v=((1.0-CONS_W)*base_v+CONS_W*meta_v).astype(np.float32)
    return {"base_train":base_o,"meta_train":meta_o,"cons_train":cons_o,
            "base_val":base_v,"meta_val":meta_v,"cons_val":cons_v}

print("Nested frozen anchor helper ready")

STATE_NAMES=np.asarray(["S0","S1","S2","S3"],dtype=object)


def state_codes(pre_anchor,v30,v31):
    pre_top=top5_indices(pre_anchor); v30_top=top5_indices(v30); v31_top=top5_indices(v31)
    a=np.any(pre_top!=v30_top,axis=1)
    b=np.any(v30_top!=v31_top,axis=1)
    return (2*a.astype(np.int8)+b.astype(np.int8)).astype(np.int8)


def top5_sha256(scores):
    top=top5_indices(scores).astype(np.int16)
    return hashlib.sha256(top.tobytes()).hexdigest()


def current066020_surface(v30,v31,state):
    v30=np.asarray(v30,np.float32); v31=np.asarray(v31,np.float32); state=np.asarray(state,np.int8)
    t30=top5_indices(v30); t31=top5_indices(v31)
    top1=(t30[:,0]!=t31[:,0])
    membership=np.asarray([set(a)!=set(b) for a,b in zip(t30,t31)],bool)
    s3=(state==3)
    keep=s3 & (top1 | membership)
    out=v30.copy(); out[keep]=v31[keep]
    info={
        "s3":s3,
        "top1":s3 & top1,
        "membership":s3 & membership & ~top1,
        "order_only":s3 & ~top1 & ~membership,
        "accepted":keep,
    }
    return out,info


def classify_single_adjacent_order_swap(v30_top_row,v31_top_row):
    a=np.asarray(v30_top_row,int); b=np.asarray(v31_top_row,int)
    if a[0]!=b[0] or set(a)!=set(b):
        raise AssertionError(("not pure order-only",a.tolist(),b.tolist()))
    diff=np.flatnonzero(a!=b)
    if len(diff)!=2 or diff[1]!=diff[0]+1:
        raise AssertionError(("not a single adjacent transposition",a.tolist(),b.tolist(),diff.tolist()))
    lo,hi=int(diff[0]),int(diff[1])
    if not (a[lo]==b[hi] and a[hi]==b[lo]):
        raise AssertionError(("not a two-way swap",a.tolist(),b.tolist()))
    boundary=f"B{lo+1}{hi+1}"
    if boundary not in EXPECTED_BOUNDARY_COUNTS:
        raise AssertionError(("unexpected order-only boundary",boundary,a.tolist(),b.tolist()))
    promoted=int(b[lo])
    demoted=int(a[lo])
    return boundary,promoted,demoted,lo+1,hi+1


def submission_from_scores(scores):
    out=sample.copy(); out.loc[:,MODULES]=np.asarray(scores,np.float32)
    assert out.shape==sample.shape
    assert out["user_id"].astype(str).tolist()==test["user_id"].astype(str).tolist()
    assert np.isfinite(out[MODULES].to_numpy()).all()
    return out


def candidate_geometry(name,anchor,candidate):
    ta=top5_indices(anchor); tc=top5_indices(candidate)
    changed=np.any(ta!=tc,axis=1)
    membership=np.asarray([set(a)!=set(b) for a,b in zip(ta,tc)],bool)
    return {
        "candidate":name,
        "changed_users_vs_current":int(changed.sum()),
        "top1_changes":int(np.sum(ta[:,0]!=tc[:,0])),
        "membership_changes":int(membership.sum()),
        "top5_sha256":top5_sha256(candidate),
    }

print("V39.5 state machine + causal boundary helpers ready")


def build_policy_anchor_split(train_rows, val_rows, seed):
    """Fit the frozen structural lineage only on train_rows and predict arbitrary global val_rows."""
    train_rows = np.asarray(train_rows, int)
    val_rows = np.asarray(val_rows, int)
    anc = build_anchor_split_full(train_rows, val_rows, int(seed))

    # Intent-State training features are strictly cross-fitted inside train_rows.
    intent_tr = crossfit_intentstate_features(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows],
        MODULE_READY_BAND[train_rows], TRAIN_GROUP[train_rows], int(seed)+71
    )
    bundle = fit_intentstate_bundle(
        y[train_rows], ORDINAL_GRADES[train_rows], TRAIN_INTENT[train_rows], MODULE_READY_BAND[train_rows]
    )
    intent_va = apply_intentstate_bundle(bundle, ALL_INTENT[val_rows], MODULE_READY_BAND[val_rows])

    Xtr, names, bounds = build_long_matrix_v31(
        train_rows, anc["cons_train"], anc["base_train"], anc["meta_train"], intent_tr
    )
    Xva, _, bounds2 = build_long_matrix_v31(
        val_rows, anc["cons_val"], anc["base_val"], anc["meta_val"], intent_va
    )
    assert bounds == bounds2
    be = bounds["base"]
    ie = bounds["intentstate"]
    _, raw30 = fit_ranker_predict(Xtr[:,:be], Xva[:,:be], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+101)
    _, raw31 = fit_ranker_predict(Xtr[:,:ie], Xva[:,:ie], ORDINAL_GRADES[train_rows], len(train_rows), int(seed)+102)
    v30 = blend_residual(anc["cons_val"], raw30, ALPHA)
    v31 = blend_residual(anc["cons_val"], raw31, ALPHA)
    st = state_codes(anc["cons_val"], v30, v31)
    current, info = current066020_surface(v30, v31, st)

    # Apply the frozen B34/B45 structural policy mechanically, never by user id.
    t30 = top5_indices(v30); t31 = top5_indices(v31)
    for r in np.flatnonzero(info["order_only"]):
        try:
            boundary, *_ = classify_single_adjacent_order_swap(t30[r], t31[r])
        except AssertionError:
            continue
        if boundary in {"B34", "B45"}:
            current[r] = v31[r]

    # Training structural surface remains purely cross-fitted inside train_rows.
    train_anchor = anc["cons_train"].copy()
    out = {
        "train_anchor": train_anchor,
        "val_anchor": current.astype(np.float32),
        "val_state": st.astype(np.int8),
        "base_train": anc["base_train"], "meta_train": anc["meta_train"],
        "base_val": anc["base_val"], "meta_val": anc["meta_val"],
    }
    del Xtr, Xva, intent_tr, intent_va, bundle, anc, raw30, raw31
    gc.collect()
    return out

print("policy-matched validation anchor helper ready")

print("V42 policy engine initialized; explicit policy_anchor role ready")

POLICY_CACHE=OUT/"policy_anchor_cache"; POLICY_CACHE.mkdir(parents=True,exist_ok=True)
def cached_policy_anchor_split(train_rows,val_rows,seed):
    tr=np.asarray(train_rows,np.int32); va=np.asarray(val_rows,np.int32)
    h=hashlib.sha256(); h.update(b"V131-V42-POLICY-CACHE-v1"); h.update(np.asarray([int(seed)],np.int64).tobytes()); h.update(tr.tobytes()); h.update(b"|"); h.update(va.tobytes())
    f=POLICY_CACHE/(h.hexdigest()+".npz")
    if f.exists():
        z=np.load(f); a=np.asarray(z["val_anchor"],np.float32); st=np.asarray(z["val_state"],np.int8)
        if a.shape==(len(va),M) and st.shape==(len(va),): return {"val_anchor":a,"val_state":st}
    z=build_policy_anchor_split(tr,va,int(seed)); np.savez_compressed(f,val_anchor=z["val_anchor"],val_state=z["val_state"])
    return {"val_anchor":np.asarray(z["val_anchor"],np.float32),"val_state":np.asarray(z["val_state"],np.int8)}
print("policy-anchor cache",POLICY_CACHE)

invalid assessment cells: 251 / 50000
administrative/noise rows: 535 / 14573
USER_FEATURES: (5000, 134)
example augmented text: [older] min aku mau fokus jadi data analyst aja deh mau kuasain excel sama sql tpl_93c96e6b92 topic_excel topic_sql [recent] eh min gajadi deh bosku nyuruh pindah haluan bikin ai ada kelas genai nggak tpl_abc6b29bac topic_genai recent_topic_genai recent_topic_genai [recent] sori nanya mulu ya min tapi kelas yg cover data cleaning ada silabus lengkapnya ga tpl_ace230065e topic_eda recent_topic_eda recent_topic_eda [current] sori nanya mulu ya min tapi kelas belajar pyton dari nol itu berapa lama 
catalog lexical matrices: (5000, 17) (5000, 17)
latest template groups: 1095
latest intent signatures train/test: 46 46
test intent signatures unseen in train: 0
readiness band counts: {np.int8(0): np.int64(45948), np.int8(1): np.int64(16376), np.int8(2): np.int64(5676)}
ordinal grade counts: {np.int16(0): np.int64(48009), np.int16(1): np.int64(1353), np.int16(2): np.i

In [3]:
# Fold-safe ordinal state utility G and bounded trust-region fusion.
# These features are organizer-derived only; no user-ID transformation is a predictive feature.
ID_NUM={'nol':0,'satu':1,'dua':2,'tiga':3,'empat':4,'lima':5}
_parsed={}; _questions=[]
for r in assess.itertuples(index=False):
    txt=str(r.assessment_result)
    try: d=json.loads(txt)
    except Exception:
        try: d=ast.literal_eval(txt)
        except Exception: d={}
    _parsed[str(r.user_id)]=d
    if not _questions and len(d)>=10: _questions=list(d.keys())[:10]
assert len(_questions)==10
ASS_RAW=np.full((N,10),np.nan,np.float32)
ASS_FLAG=np.ones((N,10),np.float32)
ASS_STRING=np.zeros((N,10),np.float32)
for i,u in enumerate(ALL_USERS):
    d=_parsed.get(str(u),{})
    for j,q in enumerate(_questions):
        v=d.get(q,np.nan); flagged=0.; isstr=float(isinstance(v,str))
        if isinstance(v,str):
            vv=v.strip().lower(); v=ID_NUM.get(vv,vv); flagged=1.
        try: x=float(v)
        except Exception: x=np.nan; flagged=1.
        if not np.isfinite(x) or x<0 or x>5: x=np.nan; flagged=1.
        ASS_RAW[i,j]=x; ASS_FLAG[i,j]=flagged; ASS_STRING[i,j]=isstr

# Label-free chat state. LATEST_GROUP is supplied by the organizer-only anchor parser above.
chat_state_src=chat.copy()
chat_state_src['timestamp']=pd.to_datetime(chat_state_src['timestamp'],errors='coerce')
chat_state_src=chat_state_src[chat_state_src.user_id.astype(str).isin(set(ALL_USERS))].sort_values(['user_id','timestamp','chat_id'])
_by={str(u):g for u,g in chat_state_src.groupby(chat_state_src.user_id.astype(str),sort=False)}
_valid_ts=chat_state_src.timestamp.dropna(); _origin=_valid_ts.min() if len(_valid_ts) else pd.Timestamp('2026-01-01')
CHAT_NUM=np.zeros((N,9),np.float32)  # logn, low, med, high, no-chat, absolute-first-day, span, present, pivot
for i,u in enumerate(ALL_USERS):
    g=_by.get(str(u)); n=0 if g is None else len(g); ts=[] if g is None else [t for t in g.timestamp.tolist() if not pd.isna(t)]
    low=float(1<=n<=2); med=float(3<=n<=5); high=float(n>=6); nochat=float(n==0); present=float(n>0)
    first=float(ts[0].timestamp()/86400.0) if ts else np.nan; span=float((ts[-1]-ts[0]).total_seconds()/86400.0) if len(ts)>=2 else 0.
    pivot=0.
    if n in (3,4,5) and len(ts)>=3:
        gd=np.diff(np.array([t.timestamp() for t in ts],float))/86400.0
        pivot=float(25<=gd[0]<=60 and np.all(gd[1:]>=3))
    CHAT_NUM[i]=[np.log1p(n),low,med,high,nochat,first,span,present,pivot]

STATE_CORE_END=None

def fit_state_bundle(ref):
    ref=np.asarray(ref,int)
    med=np.nanmedian(ASS_RAW[ref],axis=0); med=np.where(np.isfinite(med),med,0.0).astype(np.float32)
    ai=np.where(np.isnan(ASS_RAW[ref]),med,ASS_RAW[ref]).astype(np.float32)
    sc=StandardScaler().fit(ai); az=sc.transform(ai).astype(np.float32)
    km=KMeans(n_clusters=6,n_init=20,random_state=20260909).fit(az)
    counts=Counter(np.asarray(LATEST_GROUP,dtype=object)[ref].tolist())
    first_raw=CHAT_NUM[ref,5]; good=np.isfinite(first_raw); origin=float(np.min(first_raw[good])) if good.any() else 0.0
    return {'med':med,'sc':sc,'km':km,'counts':counts,'time_origin':origin}

def transform_state(bundle,rows,core_only=False):
    rows=np.asarray(rows,int); med=bundle['med']
    raw=np.where(np.isnan(ASS_RAW[rows]),med,ASS_RAW[rows]).astype(np.float32)
    z=bundle['sc'].transform(raw).astype(np.float32)
    dist=bundle['km'].transform(z).astype(np.float32); lab=np.argmin(dist,axis=1)
    one=np.eye(6,dtype=np.float32)[lab]
    valid=np.where(np.isnan(ASS_RAW[rows]),np.nan,ASS_RAW[rows]); vmean=np.nanmean(valid,axis=1); vmean=np.where(np.isfinite(vmean),vmean,0.).astype(np.float32)
    bad=ASS_FLAG[rows].sum(1).astype(np.float32); strings=ASS_STRING[rows].sum(1).astype(np.float32)
    core=np.column_stack([raw,ASS_FLAG[rows],z,dist,one,vmean,bad,strings]).astype(np.float32)
    if core_only: return core
    chatx=CHAT_NUM[rows].copy(); present=chatx[:,7]>0; chatx[:,5]=np.where(present,chatx[:,5]-float(bundle['time_origin']),0.0)
    tf=np.asarray([np.log1p(bundle['counts'].get(str(LATEST_GROUP[r]),0)) for r in rows],np.float32)[:,None]
    return np.hstack([core,chatx,tf]).astype(np.float32)

def fit_state_predict(ref,query,core_only=False,q12=False):
    ref=np.asarray(ref,int); query=np.asarray(query,int); bun=fit_state_bundle(ref)
    X=transform_state(bun,ref,core_only=core_only); Q=transform_state(bun,query,core_only=core_only)
    if not q12:
        model=Ridge(alpha=1000.0,solver='lsqr').fit(X,ORDINAL_GRADES[ref].astype(np.float32))
        pred=model.predict(Q).astype(np.float32)
    else:
        target=ORDINAL_GRADES[ref].astype(np.float32); mu=target.mean(0,keepdims=True); cen=target-mu
        _,_,vt=np.linalg.svd(cen,full_matrices=False); V=vt[:12].T.astype(np.float32); latent=cen@V
        model=Ridge(alpha=100.0,solver='lsqr').fit(X,latent)
        pred=(mu+model.predict(Q)@V.T).astype(np.float32)
    return pred,bun

def row_z(S):
    S=np.asarray(S,np.float32); return (S-S.mean(1,keepdims=True))/np.maximum(S.std(1,keepdims=True),1e-9)

def apply_trust_region(B,G,alpha):
    B=np.asarray(B,np.float32); G=np.asarray(G,np.float32); assert B.shape==G.shape and B.shape[1]==17
    f=row_z(B)+float(alpha)*row_z(G); out=B.copy(); bo=np.argsort(-B,axis=1,kind='stable')
    for i in range(len(B)):
        pool=bo[i,1:8].copy(); key=f[i,pool]; pool=pool[np.argsort(-key,kind='stable')]
        new_order=np.r_[bo[i,0],pool,bo[i,8:]]
        vals=np.linspace(1.0,0.0,M,dtype=np.float32)
        out[i,new_order]=vals
    assert np.array_equal(np.argmax(out,axis=1),np.argmax(B,axis=1)), 'Top1 authority violation'
    bo2=np.argsort(-B,axis=1,kind='stable'); co2=np.argsort(-out,axis=1,kind='stable')
    assert np.array_equal(bo2[:,8:],co2[:,8:]), 'tail ranks 9-17 changed'
    return out

def choose_alpha(y_local,B,G):
    vals=[]
    for a in ALPHA_GRID:
        C=apply_trust_region(B,G,a); vals.append((a,ndcg5(y_local,C)))
    mx=max(v for _,v in vals); eligible=[a for a,v in vals if mx-v<=1e-5]
    return float(min(eligible)),vals

def membership_change_rate(B,C):
    a=top5_idx(B); b=top5_idx(C); return float(np.mean([set(x)!=set(z) for x,z in zip(a,b)]))

def rank_change_ledger(rows,B,C,G,fold):
    bo=np.argsort(-B,axis=1,kind='stable'); co=np.argsort(-C,axis=1,kind='stable'); rec=[]
    for j,r in enumerate(np.asarray(rows,int)):
        if np.array_equal(bo[j],co[j]): continue
        rec.append({'row':int(r),'user_id':str(ALL_USERS[r]),'fold':int(fold),'anchor_top1':MODULES[bo[j,0]],'candidate_top1':MODULES[co[j,0]],
                    'anchor_top5':'|'.join(MODULES[k] for k in bo[j,:5]),'candidate_top5':'|'.join(MODULES[k] for k in co[j,:5]),
                    'membership_changed':int(set(bo[j,:5])!=set(co[j,:5]))})
    return rec


In [4]:
# Fully nested primary random CV: B inner-OOF for alpha selection, outer-paired evaluation.
PRIMARY_SPLITS=random_outer_splits()
OOF_B=np.zeros((NTR,M),np.float32); OOF_G=np.zeros_like(OOF_B); OOF_C=np.zeros_like(OOF_B)
OOF_CORE=np.zeros_like(OOF_B); OOF_Q12=np.zeros_like(OOF_B); FOLD_ID=np.full(NTR,-1,np.int16)
PRIMARY_ROWS=[]; INNER_ROWS=[]; CHANGE_ROWS=[]; SELECTED_ALPHAS=[]

def anchor_oof_for_outer_train(rows,seed):
    rows=np.asarray(rows,int); pos={g:i for i,g in enumerate(rows)}; B=np.zeros((len(rows),M),np.float32)
    inn=inner_splits(rows,seed)
    for k,(a,b) in enumerate(inn):
        z=cached_policy_anchor_split(a,b,int(seed)+100+k)
        for j,g in enumerate(b): B[pos[g]]=z['val_anchor'][j]
    assert np.isfinite(B).all(); return B,inn

def state_oof_for_outer_train(rows,inn):
    rows=np.asarray(rows,int); pos={g:i for i,g in enumerate(rows)}; G=np.zeros((len(rows),M),np.float32)
    for a,b in inn:
        p,_=fit_state_predict(a,b)
        for j,g in enumerate(b): G[pos[g]]=p[j]
    assert np.isfinite(G).all(); return G

for fold,(tr,va) in enumerate(PRIMARY_SPLITS):
    t0=time.time(); seed=SEED+fold*1000
    Btr,inn=anchor_oof_for_outer_train(tr,seed+10); Gtr=state_oof_for_outer_train(tr,inn)
    alpha,curve=choose_alpha(y[tr],Btr,Gtr); SELECTED_ALPHAS.append(alpha)
    for a,s in curve: INNER_ROWS.append({'fold':fold,'alpha':float(a),'inner_oof_ndcg':float(s),'selected':int(a==alpha)})
    Bva=cached_policy_anchor_split(tr,va,seed+20)['val_anchor']; Gva,_=fit_state_predict(tr,va)
    if fold==0:
        # Reproducibility guard: rerun the exact same query batch twice.
        # Do not compare against a prediction produced with a different batch shape: BLAS
        # kernels can differ at the last bits across batch sizes without implying RNG drift.
        _qchk=va[:min(64,len(va))]
        Gchk1,_=fit_state_predict(tr,_qchk); Gchk2,_=fit_state_predict(tr,_qchk)
        assert np.array_equal(np.argsort(-Gchk1,axis=1,kind='stable'),np.argsort(-Gchk2,axis=1,kind='stable')), 'state-head rank reproducibility failure'
        assert np.allclose(Gchk1,Gchk2,rtol=1e-6,atol=1e-7), 'state-head score reproducibility failure'
    Cva=apply_trust_region(Bva,Gva,alpha)
    Gcore,_=fit_state_predict(tr,va,core_only=True); Gq12,_=fit_state_predict(tr,va,q12=True)
    Ccore=apply_trust_region(Bva,Gcore,alpha); Cq12=apply_trust_region(Bva,Gq12,alpha)
    nb=per_user_ndcg(y[va],Bva); nc=per_user_ndcg(y[va],Cva)
    OOF_B[va]=Bva; OOF_G[va]=Gva; OOF_C[va]=Cva; OOF_CORE[va]=Ccore; OOF_Q12[va]=Cq12; FOLD_ID[va]=fold
    row={'fold':fold,'n':len(va),'alpha':alpha,'anchor_ndcg':float(nb.mean()),'candidate_ndcg':float(nc.mean()),'delta':float((nc-nb).mean()),
         'core_delta':float((per_user_ndcg(y[va],Ccore)-nb).mean()),'q12_delta':float((per_user_ndcg(y[va],Cq12)-nb).mean()),
         'membership_change_rate':membership_change_rate(Bva,Cva),'runtime_sec':float(time.time()-t0)}
    PRIMARY_ROWS.append(row); CHANGE_ROWS.extend(rank_change_ledger(va,Bva,Cva,Gva,fold)); print(row)

assert np.all(FOLD_ID>=0) and np.isfinite(OOF_B).all() and np.isfinite(OOF_G).all() and np.isfinite(OOF_C).all()
PRIMARY=pd.DataFrame(PRIMARY_ROWS); INNER=pd.DataFrame(INNER_ROWS); CHANGES=pd.DataFrame(CHANGE_ROWS)
PRIMARY.to_csv(OUT/'v137_primary_fold_metrics.csv',index=False); INNER.to_csv(OUT/'v137_inner_selection.csv',index=False); CHANGES.to_csv(OUT/'v137_rank_change_ledger.csv',index=False)
np.savez_compressed(OUT/'v137_oof_scores.npz',B=OOF_B,G=OOF_G,candidate=OOF_C,core=OOF_CORE,q12=OOF_Q12,fold_id=FOLD_ID)
print(PRIMARY); print('selected alphas',SELECTED_ALPHAS)


{'fold': 0, 'n': 800, 'alpha': 0.05, 'anchor_ndcg': 0.6655030603953952, 'candidate_ndcg': 0.6667014545853676, 'delta': 0.001198394189972298, 'core_delta': 0.0012770462635939975, 'q12_delta': 0.001110480110573108, 'membership_change_rate': 0.095, 'runtime_sec': 127.74614930152893}
{'fold': 1, 'n': 800, 'alpha': 0.05, 'anchor_ndcg': 0.6623091894276649, 'candidate_ndcg': 0.6634080739662204, 'delta': 0.00109888453855558, 'core_delta': 0.001973440509951617, 'q12_delta': 0.0012087773195852464, 'membership_change_rate': 0.11375, 'runtime_sec': 124.62550711631775}
{'fold': 2, 'n': 800, 'alpha': 0.05, 'anchor_ndcg': 0.6657497343478902, 'candidate_ndcg': 0.6652009883310819, 'delta': -0.0005487460168082378, 'core_delta': -0.00029824576759441076, 'q12_delta': -0.0005381649455686598, 'membership_change_rate': 0.1, 'runtime_sec': 125.63471150398254}
{'fold': 3, 'n': 800, 'alpha': 0.05, 'anchor_ndcg': 0.6671015420488803, 'candidate_ndcg': 0.6675742549880109, 'delta': 0.00047271293913066367, 'core_del

In [5]:
# Primary paired diagnostics and the first frozen gate chain.
def bootstrap_ci(delta,B=2000,seed=SEED):
    rng=np.random.default_rng(seed); n=len(delta); vals=np.empty(B,float)
    for i in range(B): vals[i]=np.mean(delta[rng.integers(0,n,n)])
    return [float(np.quantile(vals,.025)),float(np.quantile(vals,.975))]
def cluster_bootstrap_ci(delta,groups,B=2000,seed=SEED):
    rng=np.random.default_rng(seed); groups=np.asarray(groups,object); uq=np.unique(groups); mp={g:np.flatnonzero(groups==g) for g in uq}; vals=[]
    for _ in range(B):
        gs=rng.choice(uq,size=len(uq),replace=True); idx=np.concatenate([mp[g] for g in gs]); vals.append(float(np.mean(delta[idx])))
    return [float(np.quantile(vals,.025)),float(np.quantile(vals,.975))]

dB=per_user_ndcg(y,OOF_B); dC=per_user_ndcg(y,OOF_C); DELTA=dC-dB
pos_mass=float(DELTA[DELTA>0].sum()); neg_mass=float(-DELTA[DELTA<0].sum()); mass_ratio=pos_mass/max(neg_mass,1e-12)
user_ci=bootstrap_ci(DELTA,B=VCFG.bootstraps); cluster_ci=cluster_bootstrap_ci(DELTA,np.asarray(LATEST_GROUP[:NTR],object),B=VCFG.bootstraps,seed=SEED+1)
med=float(np.median(SELECTED_ALPHAS)); DEPLOY_ALPHA=max(a for a in ALPHA_GRID if a<=med+1e-12)
primary_summary={'anchor_mean':float(dB.mean()),'candidate_mean':float(dC.mean()),'gain':float(DELTA.mean()),'positive_folds':int((PRIMARY.delta>0).sum()),
                 'worst_fold':float(PRIMARY.delta.min()),'user_ci':user_ci,'cluster_ci':cluster_ci,'positive_mass':pos_mass,'negative_mass':neg_mass,'mass_ratio':mass_ratio,
                 'membership_change_rate':membership_change_rate(OOF_B,OOF_C),'selected_alphas':[float(x) for x in SELECTED_ALPHAS],'deploy_alpha':float(DEPLOY_ALPHA)}
PRIMARY_GATES={
 'primary_gain_ge_00100':primary_summary['gain']>=0.00100,
 'positive_folds_ge4':primary_summary['positive_folds']>=4,
 'worst_fold_ge_minus00050':primary_summary['worst_fold']>=-0.00050,
 'user_ci_lower_gt_minus00020':user_ci[0]>-0.00020,
 'cluster_ci_lower_gt_minus00020':cluster_ci[0]>-0.00020,
 'positive_mass_ratio_ge_1p25':mass_ratio>=1.25,
 'nonzero_alpha_ge4':sum(a>0 for a in SELECTED_ALPHAS)>=4,
 'median_alpha_le_002':DEPLOY_ALPHA<=0.02,
 'membership_rate_1_to_15pct':0.01<=primary_summary['membership_change_rate']<=0.15,
}
json.dump(primary_summary,open(OUT/'v137_primary_summary.json','w'),indent=2); print(json.dumps(primary_summary,indent=2)); print('PRIMARY_GATES',PRIMARY_GATES)
PRIMARY_PASS=all(PRIMARY_GATES.values())


{
  "anchor_mean": 0.6647798208961772,
  "candidate_mean": 0.6657185881992806,
  "gain": 0.0009387673031036111,
  "positive_folds": 4,
  "worst_fold": -0.0005487460168082378,
  "user_ci": [
    9.711817534021926e-05,
    0.0017832279343439888
  ],
  "cluster_ci": [
    0.00010583843027348397,
    0.001768341906662937
  ],
  "positive_mass": 17.747858347689757,
  "negative_mass": 13.992789135275316,
  "mass_ratio": 1.2683574501203656,
  "membership_change_rate": 0.1015,
  "selected_alphas": [
    0.05,
    0.05,
    0.05,
    0.05,
    0.05
  ],
  "deploy_alpha": 0.05
}
PRIMARY_GATES {'primary_gain_ge_00100': False, 'positive_folds_ge4': True, 'worst_fold_ge_minus00050': False, 'user_ci_lower_gt_minus00020': True, 'cluster_ci_lower_gt_minus00020': True, 'positive_mass_ratio_ge_1p25': True, 'nonzero_alpha_ge4': True, 'median_alpha_le_002': False, 'membership_rate_1_to_15pct': True}


In [6]:
# Frozen stress worlds. No retuning after the primary result.
def template_splits():
    rows=np.arange(NTR); st=strata(rows); groups=np.asarray(LATEST_GROUP[:NTR],object)
    try:
        sg=StratifiedGroupKFold(5,shuffle=True,random_state=FOLD_SEED)
        return [(rows[a],rows[b]) for a,b in sg.split(rows,st,groups)]
    except Exception:
        gg=GroupKFold(5); return [(rows[a],rows[b]) for a,b in gg.split(rows,groups=groups)]
def contiguous_splits():
    rows=np.arange(NTR); blocks=np.array_split(rows,5); return [(np.setdiff1d(rows,b),b) for b in blocks]

def eval_fixed_split(tr,va,alpha,seed,tag):
    t0=time.time(); B=cached_policy_anchor_split(tr,va,seed)['val_anchor']; G,_=fit_state_predict(tr,va); C=apply_trust_region(B,G,alpha)
    db=per_user_ndcg(y[va],B); dc=per_user_ndcg(y[va],C)
    return {'tag':tag,'n':len(va),'anchor':float(db.mean()),'candidate':float(dc.mean()),'delta':float((dc-db).mean()),'membership_change_rate':membership_change_rate(B,C),'runtime_sec':time.time()-t0}

STRESS_ROWS=[]
if PRIMARY_PASS and VCFG.run_stress:
    for f,(tr,va) in enumerate(template_splits()): STRESS_ROWS.append(eval_fixed_split(tr,va,DEPLOY_ALPHA,SEED+30000+f,f'template_{f}'))
    for f,(tr,va) in enumerate(contiguous_splits()): STRESS_ROWS.append(eval_fixed_split(tr,va,DEPLOY_ALPHA,SEED+40000+f,f'contiguous_{f}'))
    STRESS_ROWS.append(eval_fixed_split(np.arange(3200),np.arange(3200,4000),DEPLOY_ALPHA,SEED+50000,'forward_3201_4000'))
STRESS=pd.DataFrame(STRESS_ROWS)
if len(STRESS): STRESS.to_csv(OUT/'v137_stress_metrics.csv',index=False); print(STRESS)

if len(STRESS):
    tm=STRESS[STRESS.tag.str.startswith('template_')]; ct=STRESS[STRESS.tag.str.startswith('contiguous_')]; fw=float(STRESS.loc[STRESS.tag=='forward_3201_4000','delta'].iloc[0])
    STRESS_GATES={'forward_gain_ge_00040':fw>=0.00040,'contiguous_mean_ge_00050':float(ct.delta.mean())>=0.00050,'contiguous_nonnegative_ge3':int((ct.delta>=0).sum())>=3,
                  'contiguous_min_ge_minus00100':float(ct.delta.min())>=-0.00100,'template_group_min_ge_minus00100':float(tm.delta.min())>=-0.00100,'template_group_mean_positive':float(tm.delta.mean())>0}
else:
    STRESS_GATES={k:False for k in ['forward_gain_ge_00040','contiguous_mean_ge_00050','contiguous_nonnegative_ge3','contiguous_min_ge_minus00100','template_group_min_ge_minus00100','template_group_mean_positive']}
STRESS_PASS=PRIMARY_PASS and all(STRESS_GATES.values())
print('STRESS_GATES',STRESS_GATES)

# Second fixed stochastic anchor seed, only after first-seed primary + stress gates pass. Alpha stays frozen.
SECOND_ROWS=[]
if STRESS_PASS:
    for f,(tr,va) in enumerate(PRIMARY_SPLITS): SECOND_ROWS.append(eval_fixed_split(tr,va,DEPLOY_ALPHA,SECOND_SEED+f*1000,'second_seed_'+str(f)))
SECOND=pd.DataFrame(SECOND_ROWS)
if len(SECOND): SECOND.to_csv(OUT/'v137_second_seed_metrics.csv',index=False); print(SECOND)
if len(SECOND):
    SECOND_GATES={'second_seed_mean_ge_00075':float(SECOND.delta.mean())>=0.00075,'second_seed_nonnegative_ge4':int((SECOND.delta>=0).sum())>=4,'second_seed_worst_ge_minus00075':float(SECOND.delta.min())>=-0.00075}
else:
    SECOND_GATES={'second_seed_mean_ge_00075':False,'second_seed_nonnegative_ge4':False,'second_seed_worst_ge_minus00075':False}
SECOND_PASS=STRESS_PASS and all(SECOND_GATES.values())
print('SECOND_GATES',SECOND_GATES)


STRESS_GATES {'forward_gain_ge_00040': False, 'contiguous_mean_ge_00050': False, 'contiguous_nonnegative_ge3': False, 'contiguous_min_ge_minus00100': False, 'template_group_min_ge_minus00100': False, 'template_group_mean_positive': False}
SECOND_GATES {'second_seed_mean_ge_00075': False, 'second_seed_nonnegative_ge4': False, 'second_seed_worst_ge_minus00075': False}


In [7]:
# Full organizer-train/test fit, test geometry, immutable candidate writing only after all offline gates pass.
def feature_smd(trainX,testX):
    trainX=np.asarray(trainX,float); testX=np.asarray(testX,float); mu1=trainX.mean(0); mu2=testX.mean(0); v1=trainX.var(0); v2=testX.var(0)
    den=np.sqrt((v1+v2)/2); smd=np.divide(mu2-mu1,den,out=np.zeros_like(mu1),where=den>1e-9); return smd

def rank_scores(S):
    order=np.argsort(-S,axis=1,kind='stable'); out=np.empty_like(S,np.float32); vals=np.linspace(1.0,0.0,M,dtype=np.float32)
    for i in range(len(S)): out[i,order[i]]=vals
    return out

TEST_GEOMETRY={}; CANDIDATE_PATH=None
if SECOND_PASS:
    full=np.arange(NTR); test_rows=np.arange(NTR,NTR+NTE)
    Btest=cached_policy_anchor_split(full,test_rows,SEED+90000)['val_anchor']; Gtest,bundle=fit_state_predict(full,test_rows); Ctest=apply_trust_region(Btest,Gtest,DEPLOY_ALPHA)
    Xtr_state=transform_state(bundle,full,core_only=True); Xte_state=transform_state(bundle,test_rows,core_only=True)
    smd=feature_smd(Xtr_state,Xte_state); max_smd=float(np.max(np.abs(smd)))
    counts=bundle['counts']; unseen=np.asarray([counts.get(str(LATEST_GROUP[r]),0)==0 for r in test_rows],bool)
    bo=np.argsort(-Btest,axis=1,kind='stable'); co=np.argsort(-Ctest,axis=1,kind='stable'); changed=np.any(bo!=co,axis=1)
    TEST_GEOMETRY={'max_abs_state_smd':max_smd,'unseen_template_fraction':float(unseen.mean()),'unseen_template_changed_share':float(unseen[changed].mean()) if changed.any() else 0.0,
                   'changed_users':int(changed.sum()),'top1_changes':int(np.sum(bo[:,0]!=co[:,0])),'membership_change_rate':float(np.mean([set(a[:5])!=set(b[:5]) for a,b in zip(bo,co)])),
                   'no_explicit_unseen_template_indicator':True}
    TEST_GATE={'test_core_smd_lt_1p5':max_smd<1.5,'no_unseen_template_bucket_controls_predictions':True,'test_top1_immutable':TEST_GEOMETRY['top1_changes']==0}
    if all(TEST_GATE.values()):
        sub=sample.copy(); sub.loc[:,MODULES]=rank_scores(Ctest); CANDIDATE_PATH=OUT/'submission_v137_AOSUTR.csv'; sub.to_csv(CANDIDATE_PATH,index=False)
        assert sub.user_id.astype(str).tolist()==test.user_id.astype(str).tolist() and np.isfinite(sub[MODULES].to_numpy()).all()
    print('TEST_GEOMETRY',TEST_GEOMETRY,'TEST_GATE',TEST_GATE,'candidate',CANDIDATE_PATH)
else:
    TEST_GATE={'test_core_smd_lt_1p5':False,'no_unseen_template_bucket_controls_predictions':False,'test_top1_immutable':False}

ALL_GATES={**PRIMARY_GATES,**STRESS_GATES,**SECOND_GATES,**TEST_GATE}
DECISION='SUBMIT' if all(ALL_GATES.values()) and CANDIDATE_PATH is not None else ('HOLD' if all([np.isfinite(OOF_B).all(),np.isfinite(OOF_C).all()]) else 'STOP')
gate_report={'decision':DECISION,'primary':primary_summary,'primary_gates':PRIMARY_GATES,'stress_gates':STRESS_GATES,'second_seed_gates':SECOND_GATES,'test_geometry':TEST_GEOMETRY,'test_gates':TEST_GATE,
             'candidate_file':None if CANDIDATE_PATH is None else str(CANDIDATE_PATH),'submit_instruction':('SUBMIT '+str(CANDIDATE_PATH) if DECISION=='SUBMIT' else 'DO NOT SPEND A SUBMISSION SLOT')}
json.dump(gate_report,open(OUT/'v137_gate_report.json','w'),indent=2); print(json.dumps(gate_report,indent=2))


{
  "decision": "HOLD",
  "primary": {
    "anchor_mean": 0.6647798208961772,
    "candidate_mean": 0.6657185881992806,
    "gain": 0.0009387673031036111,
    "positive_folds": 4,
    "worst_fold": -0.0005487460168082378,
    "user_ci": [
      9.711817534021926e-05,
      0.0017832279343439888
    ],
    "cluster_ci": [
      0.00010583843027348397,
      0.001768341906662937
    ],
    "positive_mass": 17.747858347689757,
    "negative_mass": 13.992789135275316,
    "mass_ratio": 1.2683574501203656,
    "membership_change_rate": 0.1015,
    "selected_alphas": [
      0.05,
      0.05,
      0.05,
      0.05,
      0.05
    ],
    "deploy_alpha": 0.05
  },
  "primary_gates": {
    "primary_gain_ge_00100": false,
    "positive_folds_ge4": true,
    "worst_fold_ge_minus00050": false,
    "user_ci_lower_gt_minus00020": true,
    "cluster_ci_lower_gt_minus00020": true,
    "positive_mass_ratio_ge_1p25": true,
    "nonzero_alpha_ge4": true,
    "median_alpha_le_002": false,
    "membership

In [8]:
# Reproducibility manifest and static artifact hashes.
config={'seed':SEED,'second_seed':SECOND_SEED,'state_seed':STATE_SEED,'fold_seed':FOLD_SEED,'alpha_grid':ALPHA_GRID,'state_ridge_alpha':VCFG.state_ridge_alpha,
        'q12_ridge_alpha':VCFG.q12_ridge_alpha,'kmeans_clusters':VCFG.kmeans_clusters,'kmeans_n_init':VCFG.kmeans_n_init,'authority':'Top1 fixed; only anchor ranks 2-8 may reorder'}
manifest={'config':config,'python':platform.python_version(),'platform':platform.platform(),'data_dir':str(DATA),'inputs':{},'outputs':{},'packages':{}}
for name in REQUIRED:
    p=DATA/name; manifest['inputs'][name]={'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
try:
    import importlib.metadata as im
    manifest['packages']={x:im.version(x) for x in ['numpy','pandas','scikit-learn','scipy','lightgbm']}
except Exception: pass
for p in sorted(OUT.iterdir()):
    if p.is_file(): manifest['outputs'][p.name]={'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
json.dump(manifest,open(OUT/'v137_manifest.json','w'),indent=2)
print('OUTPUT DIR',OUT); print('DECISION',DECISION); print(json.dumps(manifest,indent=2))


OUTPUT DIR /kaggle/working/v137
DECISION HOLD
{
  "config": {
    "seed": 20260907,
    "second_seed": 20260917,
    "state_seed": 20260909,
    "fold_seed": 20260907,
    "alpha_grid": [
      0.0,
      0.01,
      0.02,
      0.05
    ],
    "state_ridge_alpha": 1000.0,
    "q12_ridge_alpha": 100.0,
    "kmeans_clusters": 6,
    "kmeans_n_init": 20,
    "authority": "Top1 fixed; only anchor ranks 2-8 may reorder"
  },
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "data_dir": "/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026",
  "inputs": {
    "chat_history.csv": {
      "bytes": 2112516,
      "sha256": "c87e93e43eb96870cc0d6b38dc7b865fbfef80de5a648890a8b5807846b9637f"
    },
    "modules_catalog.csv": {
      "bytes": 2969,
      "sha256": "b1e5320f227647e2e5745294fa6a8ba8e631a1ef436387a07518c02fc29972e5"
    },
    "sample_submission.csv": {
      "bytes": 75110,
      "sha256": "e2c99b721522ce334ccc4304be86795